# Демо 2. Шаблони розмови і підготовка даних для SFT

В демо 1 донавчання моделі було повʼязане з заміною голови, а отже частковою зміною архітектури нейронів. В багатьох випадках при донавчанні LLM моделей архітектура не змінюється, а змінюється радше формат введених і виведених данних. Для цього потрібно зрозуміти, **у якому вигляді модель бачить діалог**.
В будь-якому випадку фізично на вхід моделі подається тензор з токенами, а за те як з цих токенів утворюються семантичні повідомлення відповідають **шаблони розмови** (chat templates).
Під час донавчання модель може вивчити новий шаблон розмови.

Донавчання LLM методом SFT виконується на діалогах, відформатованих шаблоном розмови.
У цьому ноутбуці розглянемо, що таке шаблон розмови і як підготувати дані у форматі діалогів;
саме донавчання виконаємо в демо 3.

План:
1. Завантажити base- та instruct-версії SmolLM3.
2. Подивитися, як шаблон розмови перетворює різні типи діалогів на текст.
3. Порівняти відповіді base- та instruct-моделі.
4. Перевірити режими міркування `/think` і `/no_think`.
5. Порівняти шаблони різних моделей.
6. Підготувати датасет для SFT у форматі `messages`.

Демо за мотивами вправ курсу [smol-course](https://huggingface.co/learn/smol-course) і розділу 11
[Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter11/1).
Для запуску в Colab оберіть середовище з GPU: *Runtime → Change runtime type → T4 GPU*.

### Службова клітинка: таймер виконання

Ця клітинка потрібна для відповіді на питання про тривалість виконання: вона підключає до IPython обробник, який після кожної клітинки записує, скільки секунд вона виконувалася. Запускайте її **першою**.

In [1]:
# Службова клітинка до питання про тривалість: запамʼятовує, скільки секунд виконувалася кожна клітинка.
# Запускайте її першою, до всіх інших клітинок ноутбука.
import json
import os
import time

from IPython import get_ipython

TIMES_FILE = "cell_times_02.json"  # файл переживе перезапуск середовища в межах сесії Colab
CELL_TIMES = json.load(open(TIMES_FILE)) if os.path.exists(TIMES_FILE) else []
_started = [None]


def _pre_run(*args):
    _started[0] = time.perf_counter()


def _post_run(result=None):
    if _started[0] is None:  # саму службову клітинку не вимірюємо
        return
    raw = (getattr(getattr(result, "info", None), "raw_cell", "") or "").strip()
    CELL_TIMES.append({
        "cell": getattr(result, "execution_count", None),
        "sec": round(time.perf_counter() - _started[0], 1),
        "code": raw.splitlines()[0][:70] if raw else "",
    })
    with open(TIMES_FILE, "w") as f:
        json.dump(CELL_TIMES, f)


ip = get_ipython()
for event, callback in (("pre_run_cell", _pre_run), ("post_run_cell", _post_run)):
    for old in list(ip.events.callbacks[event]):
        if getattr(old, "__name__", "") == callback.__name__:
            ip.events.unregister(event, old)
    ip.events.register(event, callback)
print("Таймер клітинок увімкнено")

Таймер клітинок увімкнено


In [2]:
# Встановлюємо потрібні бібліотеки; -q вимикає зайвий вивід під час встановлення
!pip install -q "transformers>=4.55.0" "datasets" "accelerate"

In [3]:
import warnings

import torch
from datasets import get_dataset_split_names, load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

if torch.cuda.is_available():
    device = "cuda"
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Памʼять GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif torch.backends.mps.is_available():
    device = "mps"
    print("Apple MPS")
else:
    device = "cpu"
    warnings.warn("Обчислення на CPU: генерація буде дуже повільною, потрібна GPU")

GPU: Tesla T4
Памʼять GPU: 15.6 GB


In [4]:
# Детальна інформація про GPU NVIDIA: модель, зайнята й вільна памʼять
if device == "cuda":
    !nvidia-smi

Tue Nov 24 08:20:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             12W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Завантаження моделей

[**SmolLM3**](https://huggingface.co/blog/smollm3) — відкрита модель від Hugging Face на 3 млрд параметрів. Вона має дві версії:
- **Base** — лише після попереднього тренування: продовжує текст, але не веде діалог;
- **Instruct** (`SmolLM3-3B`) — донавчена виконувати інструкції й вести діалог.

In [5]:
base_model_name = "HuggingFaceTB/SmolLM3-3B-Base"
instruct_model_name = "HuggingFaceTB/SmolLM3-3B"

base_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
instruct_tokenizer = AutoTokenizer.from_pretrained(instruct_model_name)

# Ваги SmolLM3 збережено в bfloat16, але T4 не підтримує його апаратно,
# тому завантажуємо у float16.
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name, dtype=torch.float16, device_map="auto"
)
instruct_model = AutoModelForCausalLM.from_pretrained(
    instruct_model_name, dtype=torch.float16, device_map="auto"
)
print("Моделі завантажено")

config.json:   0%|          | 0.00/943 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/151 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

... (12 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/182 [00:00<?, ?B/s]

Моделі завантажено


## 2. Як шаблон розмови форматує діалог

Діалог задається списком повідомлень із ролями:
- `system` — інструкції для моделі (роль, стиль, обмеження);
- `user` — повідомлення користувача;
- `assistant` — відповіді моделі.

Метод `apply_chat_template` перетворює цей список на рядок зі спеціальними токенами.

Параметр `add_generation_prompt`:
- `False` — для передачі в модель завершених діалогів, тобто для **тренувальних даних**;
- `True` — додає в кінець заголовок відповіді асистента, щоб модель почала відповідати; використовується для **інференсу**.

Для SmolLM3 у системне повідомлення можна додати прапорець `/think` або `/no_think`,
який вмикає чи вимикає режим міркування.

In [6]:
# Різні типи діалогів для порівняння
conversations = {
    "simple_qa": [
        {"role": "system", "content": "/no_think"},
        {"role": "user", "content": "What is machine learning?"},
    ],
    "with_system": [
        {
            "role": "system",
            "content": "You are a helpful AI assistant specialized in explaining technical concepts clearly. /no_think",
        },
        {"role": "user", "content": "What is machine learning?"},
    ],
    "multi_turn": [
        {"role": "system", "content": "You are a math tutor. /no_think"},
        {"role": "user", "content": "What is calculus?"},
        {
            "role": "assistant",
            "content": "Calculus is a branch of mathematics that deals with rates of change and accumulation of quantities.",
        },
        {"role": "user", "content": "Can you give me a simple example?"},
    ],
    "reasoning_task": [
        {"role": "system", "content": "/think"},
        {
            "role": "user",
            "content": "Solve step by step: If a train travels 120 miles in 2 hours, what is its average speed?",
        },
    ],
}

for conv_type, messages in conversations.items():
    print(f"--- {conv_type.upper()} ---")

    # Завершений діалог: так виглядають тренувальні дані
    formatted_complete = instruct_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    # Діалог для інференсу: у кінці додано заголовок відповіді асистента
    formatted_prompt = instruct_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    print("Завершений діалог (add_generation_prompt=False):")
    print(formatted_complete)
    print("\nДля інференсу (add_generation_prompt=True):")
    print(formatted_prompt)
    print("\n" + "=" * 50 + "\n")

--- SIMPLE_QA ---
Завершений діалог (add_generation_prompt=False):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 24 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face.

<|im_start|>user
What is machine learning?<|im_end|>


Для інференсу (add_generation_prompt=True):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 24 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face.

<|im_start|>user
What is machine learning?<|im_end|>
<|im_start|>assistant
<think>

</think>



--- WITH_SYSTEM ---
Завершений діалог (add_generation_prompt=False):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 24 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant specialized in explaining technical concepts clearly.

<|im_s

**Що варто помітити:**
- Шаблон SmolLM3 сам додає до системного повідомлення блок метаданих (дата відсічення знань, поточна дата, режим міркування)
  і стандартну інструкцію, якщо власної немає. Модель бачить більше, ніж ми написали.
- Повідомлення позначені токенами `<|im_start|>роль` і закриті `<|im_end|>` (формат ChatML). Виняток — системний блок SmolLM3:
  у ньому немає `<|im_end|>`, його кінець — це початок наступного блоку (`<|im_start|>user`).
- З `add_generation_prompt=True` рядок закінчується на `<|im_start|>assistant`: з цього місця модель починає генерувати.

## 3. Base проти Instruct

Ставимо обом моделям одне питання. Base-модель отримує сирий текст без шаблону,
instruct-модель — діалог, відформатований шаблоном розмови.

In [7]:
def generate(model, tokenizer, text, max_new_tokens=150, temperature=0.7):
    """Генерує продовження тексту й повертає лише нові токени, без промпту."""
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Відкидаємо токени промпту: декодуємо лише те, що згенерувала модель
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [8]:
test_prompt = "Explain me, what is quantum computing?"

# Base-модель: сирий текст без шаблону
print("🤖 BASE-МОДЕЛЬ:")
print("-" * 50)
print(generate(base_model, base_tokenizer, test_prompt))

# Instruct-модель: діалог, відформатований шаблоном розмови
instruct_messages = [
    {"role": "system", "content": "/no_think"},
    {"role": "user", "content": test_prompt},
]
instruct_formatted = instruct_tokenizer.apply_chat_template(
    instruct_messages, tokenize=False, add_generation_prompt=True
)
print("\n" + "=" * 50)
print("💬 INSTRUCT-МОДЕЛЬ:")
print("-" * 50)
print(generate(instruct_model, instruct_tokenizer, instruct_formatted))

🤖 BASE-МОДЕЛЬ:
--------------------------------------------------


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


 I have read that it will change the world. How?
Quantum computing is a completely different way of computing. The way it works is with quantum bits (qubits) instead of bits.
Unlike classical computers, quantum computers use qubits, which can be both 0 and 1 at the same time.
This allows quantum computers to perform many calculations at the same time, which is very different from how classical computers work.
It is possible that quantum computers will be able to solve problems that are currently unsolvable by classical computers, such as simulating complex systems or breaking certain types of encryption.
It is possible that quantum computers will be able to do many things that are currently impossible. But we do not know for sure yet.
Quantum computing

💬 INSTRUCT-МОДЕЛЬ:
--------------------------------------------------
Quantum computing is a type of computing that uses the principles of quantum mechanics to perform calculations. Unlike classical computers, which use bits (0s and 1s)

## 4. Режими міркування `/think` і `/no_think`

SmolLM3 — гібридна модель: у режимі `/think` вона спочатку генерує міркування в блоці `<think>…</think>`,
а потім відповідь; у режимі `/no_think` блок міркувань порожній.

In [9]:
reasoning_prompts = [
    "What is 15 × 24? Show your work.",
    "A recipe calls for 2 cups of flour for 12 cookies. How much flour is needed for 30 cookies?",
    "If I have $50 and spend $18.75 on lunch and $12.30 on a book, how much money do I have left?",
]

for thinking_flag in ["/no_think", "/think"]:
    print(f"===== Режим: {thinking_flag} =====\n")
    for i, prompt in enumerate(reasoning_prompts, 1):
        messages = [
            {"role": "system", "content": thinking_flag},
            {"role": "user", "content": prompt},
        ]
        formatted = instruct_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        # Нижча температура робить відповіді на арифметичні задачі стабільнішими
        answer = generate(instruct_model, instruct_tokenizer, formatted, max_new_tokens=300, temperature=0.3)
        print(f"Задача {i}: {prompt}")
        print(f"Відповідь: {answer}")
        print("\n" + "-" * 50 + "\n")

===== Режим: /no_think =====

Задача 1: What is 15 × 24? Show your work.
Відповідь: To solve 15 × 24, we can use the standard multiplication algorithm.

First, let's break down the problem into smaller parts. We can think of 24 as 20 + 4. This allows us to split the multiplication into two parts:

15 × 20 + 15 × 4

Now, let's calculate each part separately.

15 × 20 = 300 (since 15 × 2 = 30, and we add a zero at the end to account for the tens place)

15 × 4 = 60 (since 15 × 2 = 30, and we double it to get 60)

Now, we add the two results together:

300 + 60 = 360

So, 15 × 24 = 360.

Alternatively, we can also use the standard multiplication algorithm directly:

```
   15
×  24
------
   60  (15 × 4)
 300   (15 × 20, shifted one position to the left)
------
 360
```

In both cases, we arrive at the same answer: 15 × 24 = 360.

--------------------------------------------------

Задача 2: A recipe calls for 2 cups of flour for 12 cookies. How much flour is needed for 30 cookies?
Відпов

## 5. У кожної моделі свій шаблон

Ролі `system`, `user`, `assistant` — це домовленість на рівні формату `messages`.
У токенах кожна модель записує їх по-своєму. Порівняємо шаблони кількох моделей.

In [10]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hi!"},
    {"role": "assistant", "content": "Hello! How can I help?"},
]

for name in ["HuggingFaceTB/SmolLM3-3B", "Qwen/Qwen3-0.6B", "microsoft/Phi-3.5-mini-instruct"]:
    tok = AutoTokenizer.from_pretrained(name)
    print(f"===== {name} =====")
    print(tok.apply_chat_template(messages, tokenize=False))
    print()

===== HuggingFaceTB/SmolLM3-3B =====
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 24 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a helpful assistant.

<|im_start|>user
Hi!<|im_end|>
<|im_start|>assistant
Hello! How can I help?<|im_end|>




config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

... (7 outputs left out)

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

===== microsoft/Phi-3.5-mini-instruct =====
<|system|>
You are a helpful assistant.<|end|>
<|user|>
Hi!<|end|>
<|assistant|>
Hello! How can I help?<|end|>
<|endoftext|>



Прапорці режиму міркування також залежать від моделі. У Qwen3, наприклад,
режим вмикається параметром `enable_thinking` у `apply_chat_template`.

In [11]:
qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
question = [{"role": "user", "content": "What is 2 + 2?"}]

for enable_thinking in [True, False]:
    print(f"===== enable_thinking={enable_thinking} =====")
    print(qwen_tokenizer.apply_chat_template(
        question, tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking
    ))

===== enable_thinking=True =====
<|im_start|>user
What is 2 + 2?<|im_end|>
<|im_start|>assistant

===== enable_thinking=False =====
<|im_start|>user
What is 2 + 2?<|im_end|>
<|im_start|>assistant
<think>

</think>




### А що з base-моделлю?

Base-модель не донавчали на діалогах, тому шаблон у її токенізаторі або відсутній, або модель ніколи не бачила його токенів у ролі розмітки діалогу.
Перед SFT шаблон для base-моделі задають самостійно — найпростіше взяти його з instruct-версії тієї самої моделі.

In [12]:
print("Шаблон є в base-токенізаторі:  ", base_tokenizer.chat_template is not None)
print("Шаблон є в instruct-токенізаторі:", instruct_tokenizer.chat_template is not None)

Шаблон є в base-токенізаторі:   False
Шаблон є в instruct-токенізаторі: True


## 6. Підготовка датасету для SFT

Для SFT дані зберігають у форматі `messages` — тобто у списку повідомлень із ролями.
Бібліотека TRL (`SFTTrainer`) сама застосовує до нього шаблон розмови під час тренування.

### SmolTalk2

**SmolTalk2** — датасет, на якому донавчали SmolLM3. Він складається з багатьох частин (splits) з різних джерел.
Список частин можна отримати без завантаження самих даних.

In [13]:
splits = get_dataset_split_names("HuggingFaceTB/smoltalk2", "SFT")
print(f"Кількість частин: {len(splits)}")
for s in splits:
    print(" -", s)

README.md:   0%|          | 0.00/23.5k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/124 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Кількість частин: 25
 - LongAlign_64k_Qwen3_32B_yarn_131k_think
 - OpenThoughts3_1.2M_think
 - aya_dataset_Qwen3_32B_think
 - multi_turn_reasoning_if_think
 - s1k_1.1_think
 - smolagents_toolcalling_traces_think
 - smoltalk_multilingual8_Qwen3_32B_think
 - smoltalk_systemchats_Qwen3_32B_think
 - table_gpt_Qwen3_32B_think
 - LongAlign_64k_context_lang_annotated_lang_6_no_think
 - Mixture_of_Thoughts_science_no_think
 - OpenHermes_2.5_no_think
 - OpenThoughts3_1.2M_no_think_no_think
 - hermes_function_calling_v1_no_think
 - smoltalk_multilingual_8languages_lang_5_no_think
 - smoltalk_smollm3_everyday_conversations_no_think
 - smoltalk_smollm3_explore_instruct_rewriting_no_think
 - smoltalk_smollm3_smol_magpie_ultra_no_think
 - smoltalk_smollm3_smol_rewrite_no_think
 - smoltalk_smollm3_smol_summarize_no_think
 - smoltalk_smollm3_systemchats_30k_no_think
 - table_gpt_no_think
 - tulu_3_sft_personas_instruction_following_no_think
 - xlam_traces_no_think
 - smoltalk_everyday_convs_reasoning_

In [14]:
# Завантажуємо лише файл однієї частини через data_files: повний датасет займає десятки гігабайт,
# а load_dataset(..., split=...) спершу перебирає файли всіх частин і працює дуже довго
split = "smoltalk_everyday_convs_reasoning_Qwen3_32B_think"
everyday = load_dataset("HuggingFaceTB/smoltalk2", data_files=f"SFT/{split}-*.parquet", split="train")
print(everyday)
print()
for message in everyday[0]["messages"]:
    print(f"[{message['role']}] {message['content'][:200]}")

SFT/smoltalk_everyday_convs_reasoning_Qw(…): reconstructing file:   0%|          |  0.00B / 6.33MB            

SFT/smoltalk_everyday_convs_reasoning_Qw(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['messages', 'chat_template_kwargs', 'source'],
    num_rows: 2057
})

[user] Hi there
[assistant] <think>
Okay, the user sent "Hi there". That's a friendly greeting. I should respond in a welcoming way. Let me check the guidelines. I need to be helpful, keep the conversation going, and maybe ask h
[user] I'm looking for a healthy breakfast idea. What's a good option?
[assistant] <think>
Okay, the user is asking for a healthy breakfast idea. Let me think about what makes a breakfast healthy. It should be balanced, providing a mix of nutrients like protein, fiber, healthy fats,


Ця частина зібрана для режиму `/think`, тому відповіді асистента містять блок міркувань `<think>…</think>`.

Як видно з назви частини, відповіді в ній згенерувала модель Qwen3-32B. Такі **синтетичні дані** від сильнішої моделі —
поширений спосіб отримати тисячі прикладів для SFT без ручного написання відповідей. Помилки моделі-генератора при цьому переходять у дані,
тому синтетичні дані фільтрують.

### Перетворення власного датасету у формат `messages`

Більшість датасетів має інший формат, наприклад колонки «питання» і «відповідь».
Перетворимо датасет задач з математики **GSM8K** у формат `messages`.

In [15]:
gsm8k = load_dataset("openai/gsm8k", "main", split="train[:100]")  # невелика вибірка для демо
print("Оригінальний приклад:", gsm8k[0])


def to_messages(examples):
    """Перетворює пари «питання — відповідь» на діалоги з ролями."""
    processed = []
    for question, answer in zip(examples["question"], examples["answer"]):
        processed.append([
            {"role": "system", "content": "You are a math tutor. Solve problems step by step."},
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ])
    return {"messages": processed}


gsm8k_messages = gsm8k.map(to_messages, batched=True, remove_columns=gsm8k.column_names)
print("\nУ форматі messages:", gsm8k_messages[0])

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Оригінальний приклад: {'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}


Map:   0%|          | 0/100 [00:00<?, ? examples/s]


У форматі messages: {'messages': [{'content': 'You are a math tutor. Solve problems step by step.', 'role': 'system'}, {'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'role': 'user'}, {'content': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72', 'role': 'assistant'}]}


### Що побачить модель під час тренування

Застосуємо шаблон розмови до підготовленого прикладу. Саме цей рядок модель отримає під час SFT.

In [16]:
text = instruct_tokenizer.apply_chat_template(
    gsm8k_messages[0]["messages"],
    tokenize=False,
    add_generation_prompt=False,  # діалог завершений: відповідь асистента вже є
)
print(text)
print(f"Довжина в токенах: {len(instruct_tokenizer(text)['input_ids'])}")

<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 24 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a math tutor. Solve problems step by step.

<|im_start|>user
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|im_end|>
<|im_start|>assistant
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72<|im_end|>

Довжина в токенах: 145


**Що варто помітити:** шаблон знову додав метадані й режим `/think`, хоча в даних їх немає,
а відповідь асистента не містить блоку міркувань. Якщо донавчати на таких прикладах, модель вчитиметься
відповідати без міркувань навіть у режимі `/think`. Тому формат даних і шаблон мають узгоджуватися з тим,
як модель використовуватиметься після донавчання.

## Відповіді на теоретичні питання (0.1)

Питання 11–21 до цього ноутбука. До відповідей додано код, який їх підтверджує. Клітинки нижче потрібно запускати **після** основної частини ноутбука: вони використовують змінні `instruct_model`, `instruct_tokenizer`, `base_model`, `conversations`, `splits`, `gsm8k` та інші, створені вище.

### Питання 11. Як довго виконувалися клітинки ноутбуку 02_chat_templates? Чи були проблеми з запуском, які?

**Відповідь.** Час виконання вимірювала службова клітинка на початку ноутбука: вона записує тривалість кожної клітинки, а підсумок виведено клітинкою нижче.

**Скільки це тривало** (безкоштовний Colab, GPU Tesla T4):
- за таймером, на момент цієї таблиці виконано 15 клітинок, разом **4,7 хв**. Найдовші: завантаження двох моделей по 3 млрд параметрів (Base і Instruct, ≈ 6 GB кожна) — 168,6 с (≈ 2,8 хв); порівняння режимів `/no_think` і `/think` (шість генерацій по 300 токенів) — 61,3 с; імпорти — 18,5 с; перше порівняння Base і Instruct — 14,1 с; `pip install` — 10,0 с; завантаження токенізаторів Qwen3 і Phi-3.5 — 5,9 с. За першого запуску цього ж ноутбука було 5,5 хв; різниця майже вся — у швидкості завантаження ваг;
- клітинки з відповідями на питання 12–21 виконуються вже після цього таймера, тому в таблицю не потрапили. Їхній час майже весь іде на генерацію (≈ 20 токенів/с на T4; для задачі про печиво `/think` — 850 токенів за 41,8 с): у питанні 17 три задачі в режимі `/think` забрали ≈ 2,5 хв (70,1 + 41,8 + 35,8 с), у режимі `/no_think` — ≈ 18 с; решта генерацій (питання 12, 13, 15, 16, 19) — разом близько 1,5 хв.

Разом ноутбук виконується приблизно за **9 хв** (оцінка: 4,7 хв — за таймером, ≈ 2,8 хв — питання 17 за його власною таблицею, ≈ 1,5 хв — решта генерацій; точнішої цифри таймер не дає, бо його підсумок виведено раніше за ці клітинки).

**Проблеми з запуском.**
1. Помилок виконання (traceback) у запуску немає: усі клітинки пройшли.
2. Попередження, які не є помилками: `You are sending unauthenticated requests to the HF Hub` — завантаження без токена Hugging Face (працює, але з нижчим лімітом запитів); `Ignoring clean_up_tokenization_spaces=True for BPE tokenizer` — `transformers` повідомляє, що для BPE-токенізатора цей параметр ігнорується; `rope_parameters['original_max_position_embeddings']` — попередження про формат конфігурації Qwen3 (на результат не впливає).
3. Потрібна GPU. Ваги SmolLM3 збережено в `bfloat16`, але T4 його апаратно не підтримує, тому в ноутбуці моделі завантажено у `float16`.
4. Вивід залежить від дня запуску: шаблон підставляє поточну дату (`Today Date: 24 November 2026`), тож рядки в секції 2 щодня трохи різняться.
5. Під час перевірки відповідей виявилося, що перші версії моїх контрольних клітинок до питань 12, 14 і 17 дають хибні результати (правило для питання 12 — 4 із 8, розбір блоків, перевірка відповіді на задачі). Це описано в самих відповідях, клітинки виправлено.

In [17]:
import pandas as pd


def timing_report(top=6):
    """Скільки часу забрали клітинки ноутбука (за даними службової клітинки з початку ноутбука)."""
    times = pd.DataFrame(CELL_TIMES)
    print(f"Виконано клітинок: {len(times)}; сумарний час їх виконання: {times['sec'].sum() / 60:.1f} хв")
    print(f"Найдовші {top} клітинок:")
    return times.sort_values("sec", ascending=False).head(top)[["cell", "sec", "code"]].reset_index(drop=True)


timing_report()

Виконано клітинок: 15; сумарний час їх виконання: 4.7 хв
Найдовші 6 клітинок:


,cell,sec,code
0,5,168.6,"base_model_name = ""HuggingFaceTB/SmolLM3-3B-Base"""
1,9,61.3,reasoning_prompts = [
2,3,18.5,import warnings
3,8,14.1,"test_prompt = ""Explain me, what is quantum com..."
4,2,10.0,# Встановлюємо потрібні бібліотеки; -q вимикає...
5,10,5.9,messages = [


### Питання 12. Для чого змінювати параметр add_generation_prompt в методі apply_chat_template() при використанні моделі? Як по згенерованим розмовам можна дізнатися значення цього параметру, яке було задано при виклику метода?

**Відповідь.** Параметр вирішує, чи додасть шаблон у кінець тексту **відкритий заголовок відповіді асистента**.

- `add_generation_prompt=False` — діалог рендериться «як є»: усі повідомлення завершені. Це потрібно для **тренувальних даних (SFT)**: відповідь асистента вже є в діалозі, модель вчиться її передбачати, і зайвий порожній заголовок у кінці був би помилкою.
- `add_generation_prompt=True` — у кінець додається `<|im_start|>assistant\n` (для SmolLM3 у режимі `/no_think` ще й порожній блок `<think>\n\n</think>\n`, див. вивід клітинки нижче). Це сигнал «тепер говориш ти» й місце, з якого модель починає генерувати. Використовується в **інференсі**, коли останнє повідомлення — від користувача. Без цього заголовка модель мусить сама «здогадатися», що настала її черга; що з цього виходить, показує другий експеримент нижче.

**Як за готовим текстом дізнатися значення.** Дивимося на кінець рядка: чи є після **останнього** `<|im_start|>` закриваючий `<|im_end|>`. Якщо ні, тобто текст закінчується заголовком асистента (і, можливо, порожнім `<think>`), — параметр був `True`; якщо останній блок закрито, — `False`.

*Чому правило саме таке.* Перша версія порівнювала **кількість** відкриттів і закриттів (відкриттів більше → `True`) і в запуску спрацювала лише в 4 з 8 випадків. Причина — особливість шаблону SmolLM3: **системний блок не закривається `<|im_end|>`** (його кінець — початок блоку `user`), тому відкриттів на одне більше навіть у завершеному діалозі, і правило завжди казало `True`. Порівняння позицій останніх `<|im_start|>` та `<|im_end|>` від цього не залежить; на всіх 8 відрендерених рядках із секції 2 воно дає правильний результат.

In [18]:
def guess_add_generation_prompt(text, start="<|im_start|>", end="<|im_end|>"):
    """True, якщо останній блок діалогу лишився відкритим: після останнього <|im_start|> немає <|im_end|>.
    (Порівнювати кількості не можна: системний блок SmolLM3 не закривається <|im_end|>.)"""
    return text.rfind(start) > text.rfind(end)


rows = []
for conv_type, msgs in conversations.items():
    rendered = {
        flag: instruct_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=flag)
        for flag in (False, True)
    }
    for flag, text in rendered.items():
        rows.append({
            "діалог": conv_type,
            "add_generation_prompt": flag,
            "кінець рядка": repr(text[-48:]),
            "вгадано за правилом": guess_add_generation_prompt(text),
        })
    # різниця між версіями — це і є «заголовок для генерації»
    if rendered[True].startswith(rendered[False]):
        print(f"{conv_type}: додано {rendered[True][len(rendered[False]):]!r}")
    else:
        print(f"{conv_type}: версії відрізняються не лише в кінці рядка")

check = pd.DataFrame(rows)
print(f"\nПравило спрацювало в {(check['add_generation_prompt'] == check['вгадано за правилом']).sum()} із {len(check)} випадків")
check

simple_qa: додано '<|im_start|>assistant\n<think>\n\n</think>\n'
with_system: додано '<|im_start|>assistant\n<think>\n\n</think>\n'
multi_turn: додано '<|im_start|>assistant\n<think>\n\n</think>\n'
reasoning_task: додано '<|im_start|>assistant\n'

Правило спрацювало в 8 із 8 випадків


,діалог,add_generation_prompt,кінець рядка,вгадано за правилом
0,simple_qa,False,'start|>user\nWhat is machine learning?<|im_en...,False
1,simple_qa,True,'m_end|>\n<|im_start|>assistant\n<think>\n\n</...,True
2,with_system,False,'start|>user\nWhat is machine learning?<|im_en...,False
3,with_system,True,'m_end|>\n<|im_start|>assistant\n<think>\n\n</...,True
4,multi_turn,False,'ser\nCan you give me a simple example?<|im_en...,False
5,multi_turn,True,'m_end|>\n<|im_start|>assistant\n<think>\n\n</...,True
6,reasoning_task,False,"'n 2 hours, what is its average speed?<|im_end...",False
7,reasoning_task,True,' average speed?<|im_end|>\n<|im_start|>assist...,True


In [19]:
def greedy(model, tokenizer, text, max_new_tokens=60):
    """Детермінована генерація; спеціальні токени не ховаємо, щоб бачити розмітку."""
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)


msgs = conversations["simple_qa"]
for flag in (True, False):
    text = instruct_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=flag)
    print(f"=== add_generation_prompt={flag} ===")
    print(repr(greedy(instruct_model, instruct_tokenizer, text)), "\n")

=== add_generation_prompt=True ===
'Machine learning is a subset of artificial intelligence that enables computers to learn from data, without being explicitly programmed. It involves training algorithms on large datasets to identify patterns, make predictions, and take actions based on those predictions.\n\nThe core idea of machine learning is to create models that can automatically improve their performance on' 

=== add_generation_prompt=False ===
'<|im_start|>assistant\n<think>\n\n</think>\nMachine learning is a subset of artificial intelligence that enables computers to learn from data, without being explicitly programmed. It involves training algorithms on large datasets to identify patterns, make predictions, and take actions based on those predictions.\n\nThe core idea of machine learning is to create models' 



**Результат експерименту.** З `add_generation_prompt=True` модель одразу починає відповідь. З `False` вона **сама дописала** заголовок і порожній блок міркувань (`<|im_start|>assistant\n<think>\n\n</think>\n`), а потім дала по суті ту саму відповідь. Тобто SmolLM3, навчена на цьому форматі, «відновлює» пропущений заголовок, але це витрачає токени бюджету й не гарантоване для інших, чутливіших до формату моделей або для режиму семплінгу. Тому для інференсу заголовок слід додавати завжди.

### Питання 13. Чим відрізняються ролі system від assistant в шаблонах розмов?

**Відповідь.** У токенах обидві ролі — лише різні заголовки (`<|im_start|>system` і `<|im_start|>assistant`), але функції різні.

- **`system`** — інструкція розробника: роль, стиль, формат відповіді, обмеження, режим міркування. Її пишуть один раз, вона стоїть на початку й діє на весь діалог; користувач зазвичай її не бачить. Модель системних повідомлень **не генерує**; її вчать їх дотримуватися.
- **`assistant`** — репліки самої моделі: історія попередніх відповідей у діалозі (щоб модель памʼятала, що вже казала) і, у тренувальних даних, **еталонна відповідь**, яку модель має навчитися відтворювати. Саме на цих токенах рахується loss при SFT з `assistant_only_loss=True`, і саме їх модель генерує в інференсі.

Різницю видно й у тому, як шаблон SmolLM3 обробляє ролі (вивід до питання 14): текст `system` він розкладає в блок `## Custom Instructions`, а прапорці `/think` і `/no_think` виносить у метадані; повідомлення `assistant` залишаються окремими блоками `<|im_start|>assistant … <|im_end|>` (у режимі `/no_think` — з порожнім `<think>` перед текстом).

Чи змінює роль поведінку моделі, перевіряємо експериментом: ту саму вимогу «відповідай одним словом» ставимо спершу як `system`, потім як попередню репліку `assistant`, на трьох питаннях; колонка «блок у рядку» показує, де шаблон розмістив вимогу.

**Результат запуску.** Усі шість відповідей — по одному слову («Paris», «Artificial», «Boil.» із вимогою в ролі `system`; «Paris», «Network», «Boil.» із вимогою в ролі `assistant`), а колонка «блок у рядку» підтверджує, що шаблон справді розмістив вимогу в блоці `system` в одному варіанті й у блоці `assistant` в іншому. Тобто на цій моделі й цих трьох питаннях слухняність від ролі не залежала: невелика Instruct-модель, імовірно, підхоплює стиль із будь-якої частини контексту, зокрема з «власної» попередньої репліки. Це не заперечує різниці між ролями, яка полягає в обробці шаблоном і в навчанні (loss рахується на токенах `assistant`, а не `system`), а не в тому, що «`system` завжди слухняніший». Три питання — це ілюстрація, а не статистика.

In [20]:
instruction = "Always answer with exactly one word."
questions = [
    "What is the capital of France and why is it famous?",
    "Explain what a neural network is.",
    "How do I make a cup of tea?",
]


def build_dialog(instruction_role, question):
    """Та сама вимога або в системному повідомленні, або як попередня репліка асистента."""
    if instruction_role == "system":
        return [{"role": "system", "content": instruction + " /no_think"}, {"role": "user", "content": question}]
    return [
        {"role": "system", "content": "/no_think"},
        {"role": "assistant", "content": instruction},
        {"role": "user", "content": question},
    ]


def block_role_with(text, phrase):
    """Роль блоку, у якому шаблон розмістив фразу."""
    for chunk in text.split("<|im_start|>")[1:]:
        if phrase in chunk:
            return chunk.split("\n", 1)[0]


rows = []
for instruction_role in ("system", "assistant"):
    for question in questions:
        rendered_text = instruct_tokenizer.apply_chat_template(
            build_dialog(instruction_role, question), tokenize=False, add_generation_prompt=True
        )
        answer = greedy(instruct_model, instruct_tokenizer, rendered_text, max_new_tokens=80).replace("<|im_end|>", "").strip()
        rows.append({
            "вимога в ролі": instruction_role,
            "блок у рядку": block_role_with(rendered_text, instruction),
            "питання": question[:36],
            "слів у відповіді": len(answer.split()),
            "відповідь": answer[:50].replace("\n", " ⏎ "),
        })
pd.DataFrame(rows)

,вимога в ролі,блок у рядку,питання,слів у відповіді,відповідь
0,system,system,What is the capital of France and wh,1,Paris
1,system,system,Explain what a neural network is.,1,Artificial
2,system,system,How do I make a cup of tea?,1,Boil.
3,assistant,assistant,What is the capital of France and wh,1,Paris
4,assistant,assistant,Explain what a neural network is.,1,Network
5,assistant,assistant,How do I make a cup of tea?,1,Boil.


### Питання 14. Прокоментуйте вивід даних в секції 2 «Як шаблон розмови форматує діалог». Які секції було додано шаблонізатором розмов і для чого?

**Відповідь.** Для кожного з чотирьох діалогів виведено два рядки: завершений діалог (`add_generation_prompt=False`) і той самий діалог для інференсу (`True`). З виводу видно, що шаблон SmolLM3 **додає значно більше тексту, ніж ми написали**:

1. **Обгортки ролей.** Повідомлення позначені `<|im_start|>роль` і закриті `<|im_end|>` (формат ChatML). Це дає моделі чіткі межі реплік і спосіб зрозуміти, хто говорить. Виняток — **системний блок: у SmolLM3 він не закривається `<|im_end|>`**, його кінець — це початок наступного блоку `<|im_start|>user`.
2. **Блок метаданих системного повідомлення** (`## Metadata`): дата відсічення знань моделі (Knowledge Cutoff Date), поточна дата (Today Date, її підставляє `strftime_now` під час рендеру, тому вона змінюється щодня) і режим міркування (Reasoning Mode: `/think` або `/no_think`). Модель навчали на системних повідомленнях із цим блоком, тож він повідомляє їй, у якому режимі працювати, і дає орієнтир щодо дати.
3. **Блок користувацьких інструкцій** (`## Custom Instructions`): це наш системний текст без прапорців `/think`/`/no_think` (шаблон їх вирізає й переносить у метадані). Якщо власного тексту немає, шаблон підставляє інструкцію за замовчуванням («You are a helpful AI assistant named SmolLM, trained by Hugging Face»; у режимі `/think` вона довша й описує формат «Thought → Solution»).
4. **Порожній блок `<think>\n\n</think>`** у режимі `/no_think`: він додається перед попередніми репліками асистента (діалог `multi_turn`) і в кінці рядка для інференсу. Так модель бачить «міркування вже завершено, їх нема» й одразу відповідає.
5. **Заголовок асистента в кінці** (`<|im_start|>assistant`) тільки в рядках для інференсу — місце, з якого модель починає генерувати (питання 12).

Клітинка нижче розбирає рядки на блоки (з урахуванням того, що системний блок не закритий), показує службові поля й підраховує, скільки токенів додав шаблон. *Перша версія розбору шукала пари `<|im_start|>…<|im_end|>` і через незакритий системний блок «з'їдала» блок `user` у системний; підрахунок токенів і метадані при цьому були правильні, а список блоків — ні. Розбір виправлено: блоки відокремлюються за `<|im_start|>`.*

**Результат запуску.** Скільки токенів додав шаблон до наших повідомлень:

| діалог | наших токенів | у рядку для інференсу | додав шаблон | режим |
|---|---|---|---|---|
| simple_qa | 8 | 73 | 65 | `/no_think` |
| with_system | 22 | 69 | 47 | `/no_think` |
| multi_turn | 40 | 101 | 61 | `/no_think` |
| reasoning_task | 26 | 272 | 246 | `/think` |

У режимі `/no_think` службовий текст дає 47–65 токенів, а в `/think` — 246: майже весь рядок (≈ 90%) — це довга інструкція за замовчуванням. Дата й дата відсічення в усіх чотирьох рядках однакові (`June 2025`, `24 November 2026`).

Розбір на блоки після виправлення дає `[system, user, assistant (відкритий)]` для трьох діалогів і `[system, user, assistant, user, assistant (відкритий)]` для `multi_turn`; «відкритий» означає заголовок асистента без `<|im_end|>`.

In [21]:
import re


def split_blocks(text):
    """Розбиває відрендерений діалог на блоки (роль, текст).
    Блок закінчується на <|im_end|> або (системний блок SmolLM3 без <|im_end|>) на початку наступного блоку.
    Відкритий блок в кінці рядка (заголовок асистента) позначається «(відкритий)»."""
    blocks = []
    chunks = text.split("<|im_start|>")[1:]
    for k, chunk in enumerate(chunks):
        role, _, body = chunk.partition("\n")
        closed = "<|im_end|>" in body
        body = body.split("<|im_end|>", 1)[0].strip("\n")
        is_open = (k == len(chunks) - 1) and not closed
        blocks.append((role + (" (відкритий)" if is_open else ""), body))
    return blocks


rows = []
for conv_type, msgs in conversations.items():
    prompt_text = instruct_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ours = sum(len(instruct_tokenizer(m["content"])["input_ids"]) for m in msgs)
    total = len(instruct_tokenizer(prompt_text)["input_ids"])
    system_block = next((t for r, t in split_blocks(prompt_text) if r == "system"), "")
    meta = {k: (re.search(rf"{k}: (.*)", system_block) or [None, "—"])[1] for k in ("Knowledge Cutoff Date", "Today Date", "Reasoning Mode")}
    rows.append({
        "діалог": conv_type, "наших токенів": ours, "токенів у рядку": total, "додав шаблон": total - ours,
        "дата відсічення": meta["Knowledge Cutoff Date"], "сьогодні": meta["Today Date"], "режим": meta["Reasoning Mode"],
        "блоки": [r for r, _ in split_blocks(prompt_text)],
    })
pd.DataFrame(rows)

,діалог,наших токенів,токенів у рядку,додав шаблон,дата відсічення,сьогодні,режим,блоки
0,simple_qa,8,73,65,June 2025,24 November 2026,/no_think,"[system, user, assistant (відкритий)]"
1,with_system,22,69,47,June 2025,24 November 2026,/no_think,"[system, user, assistant (відкритий)]"
2,multi_turn,40,101,61,June 2025,24 November 2026,/no_think,"[system, user, assistant, user, assistant (від..."
3,reasoning_task,26,272,246,June 2025,24 November 2026,/think,"[system, user, assistant (відкритий)]"


In [22]:
example = instruct_tokenizer.apply_chat_template(conversations["with_system"], tokenize=False, add_generation_prompt=True)
for role, body in split_blocks(example):
    print(f"--- {role} ---")
    print(body if body is not None else "")

--- system ---
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 24 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant specialized in explaining technical concepts clearly.
--- user ---
What is machine learning?
--- assistant (відкритий) ---
<think>

</think>


### Питання 15. Прокоментуйте різницю між Base і Instruct, проглянувши вивід в секції 3.

**Відповідь.** Обидві моделі мають однакову архітектуру й знання, здобуті під час попереднього тренування; різниця в **поведінці**, яку додає донавчання на діалогах.

- **Base** навчена лише передбачати наступний токен у великих корпусах (сторінки, книжки, код). Там після питання «Explain me, what is quantum computing?» може йти що завгодно: наступне питання зі списку, варіанти відповіді, початок статті, обговорення на форумі. Тому вона **продовжує текст**, а не відповідає, часто повторює або вигадує нові запитання й не вміє «зупинитися»: токена кінця відповіді вона не генерує, тож доходить до ліміту `max_new_tokens`.
- **Instruct** після SFT (і подальшого вирівнювання) бачила тисячі діалогів у форматі шаблону, де після ролі `user` завжди йде змістовна відповідь `assistant`, що закінчується `<|im_end|>`. Тому вона відповідає на питання по суті, звертається до користувача й **завершує відповідь сама**.

Зверніть увагу й на формат входу: Base отримала сирий текст, Instruct — діалог, відформатований шаблоном; це частина різниці. Виводи у секції 3 генеруються із `temperature=0.7`, тож від запуску до запуску вони різняться; щоб порівняння було відтворюваним, клітинка нижче використовує жадібне декодування й рахує, чи зупинилася модель сама.

**Результат запуску** (жадібне декодування, ліміт 120 токенів):

| запит | модель | токенів | завершила сама | знаків «?» у відповіді |
|---|---|---|---|---|
| quantum computing | Base | 120 | ні | 5 |
| quantum computing | Instruct | 120 | ні | 0 |
| capital of France | Base | 120 | ні | 9 |
| capital of France | Instruct | 50 | так | 0 |
| reverse a list in Python | Base | 120 | ні | 2 |
| reverse a list in Python | Instruct | 120 | ні | 0 |

Base-модель **жодного разу не зупинилася сама** і в усіх трьох випадках продовжувала текст питаннями: після «Paris.» вона дописала «What is the capital of the United States? Washington, D.C. What is the capital of t…» (9 знаків «?»), на питання про квантові обчислення відповіла ланцюжком нових питань («How it is different from classical computing? What are the advantages…»), а на питання про Python почала з повтору запиту. Instruct-модель у всіх трьох випадках **відповіла по суті й без жодного «?»**. Сама зупинилася лише в одному: на двох інших розгорнута відповідь просто не вмістилася в 120 токенів, тож їх обірвав ліміт, а не неспроможність завершити. Цифри — для одного запуску з трьома запитами, це ілюстрація, а не статистика.

In [23]:
def greedy_with_stop(model, tokenizer, text, max_new_tokens=120):
    """Жадібна генерація; повертає текст, кількість токенів і чи завершилася відповідь сама (токен кінця)."""
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    eos = model.generation_config.eos_token_id
    eos_ids = set(eos if isinstance(eos, (list, tuple)) else [eos if eos is not None else tokenizer.eos_token_id])
    stopped = any(int(t) in eos_ids for t in new_tokens)
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip(), len(new_tokens), stopped


prompts = ["Explain me, what is quantum computing?", "What is the capital of France?", "How do I reverse a list in Python?"]
rows = []
for prompt in prompts:
    base_text, base_n, base_stop = greedy_with_stop(base_model, base_tokenizer, prompt)
    chat_text = instruct_tokenizer.apply_chat_template(
        [{"role": "system", "content": "/no_think"}, {"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True
    )
    inst_text, inst_n, inst_stop = greedy_with_stop(instruct_model, instruct_tokenizer, chat_text)
    for model_name, answer, n, stop in (("Base", base_text, base_n, base_stop), ("Instruct", inst_text, inst_n, inst_stop)):
        rows.append({"запит": prompt, "модель": model_name, "токенів": n, "завершила сама": stop,
                     "знаків «?» у відповіді": answer.count("?"), "початок відповіді": answer[:90].replace("\n", " ⏎ ")})
pd.set_option("display.max_colwidth", 120)
pd.DataFrame(rows)

,запит,модель,токенів,завершила сама,знаків «?» у відповіді,початок відповіді
0,"Explain me, what is quantum computing?",Base,120,False,5,How it is different from classical computing? What are the advantages of quantum computing
1,"Explain me, what is quantum computing?",Instruct,120,False,0,Quantum computing is a type of computing that uses the principles of quantum mechanics to
2,What is the capital of France?,Base,120,False,9,"Paris. What is the capital of the United States? Washington, D.C. What is the capital of t"
3,What is the capital of France?,Instruct,50,True,0,The capital of France is Paris. Paris is a city in north-central France and is known for i
4,How do I reverse a list in Python?,Base,120,False,2,How do I reverse a list in Python? Python has a built-in function called reverse() that re
5,How do I reverse a list in Python?,Instruct,120,False,0,You can reverse a list in Python using the `reverse()` method or slicing. Here are example


### Питання 16. Чому в режимі /think при генерації було передано параметр max_new_tokens=300?

**Відповідь.** У режимі `/think` модель спершу генерує блок міркувань `<think>…</think>`, і лише потім — відповідь. Міркування витрачають токени з того самого бюджету `max_new_tokens`, тому для `/think` потрібен значно більший бюджет, ніж для `/no_think`, де блок порожній, і більший за стандартні 150 токенів функції `generate()`. Якщо бюджет замалий, генерація **обривається посеред міркувань**, і відповіді (розділу після `</think>`) користувач не отримує взагалі.

З іншого боку, ліміт потрібен, щоб міркування не затягнулися: час і памʼять (KV-кеш) ростуть із довжиною, а мала модель може зациклитися. Тому 300 — це компроміс для швидкого демо (у ньому те саме значення передано в обох режимах).

**Що показав запуск.** Для `/think` значення 300 виявилося **замалим**: у секції 4 жодна з трьох відповідей не дійшла до розділу після `</think>`, усі три обірвалися посеред міркувань. Експеримент нижче пояснює це кількісно: генеруємо відповідь на задачу про печиво один раз із запасом (1000 токенів) і «обрізаємо» її до різних бюджетів — жадібне декодування префікс-стабільне, тому це еквівалентно запускам із різним `max_new_tokens`. Результат: уся відповідь займає 850 токенів, із них міркування — ≈ 544; при бюджетах 50, 150 і 300 міркування не завершені й відповіді нема, а при 600 — завершені, і відповідь є. Тобто навіть для такої простої задачі `/think` потребує сотень токенів лише на міркування; для складніших задач бюджет має бути ще більшим.

In [24]:
think_prompt = "A recipe calls for 2 cups of flour for 12 cookies. How much flour is needed for 30 cookies?"
think_text = instruct_tokenizer.apply_chat_template(
    [{"role": "system", "content": "/think"}, {"role": "user", "content": think_prompt}], tokenize=False, add_generation_prompt=True
)
inputs = instruct_tokenizer(think_text, return_tensors="pt").to(instruct_model.device)
with torch.no_grad():
    full_ids = instruct_model.generate(**inputs, max_new_tokens=1000, do_sample=False, pad_token_id=instruct_tokenizer.eos_token_id)
new_ids = full_ids[0][inputs["input_ids"].shape[1]:]

rows = []
for budget in (50, 150, 300, 600):
    partial = instruct_tokenizer.decode(new_ids[:budget], skip_special_tokens=False)
    finished = "</think>" in partial
    answer = partial.split("</think>", 1)[1].replace("<|im_end|>", "").strip() if finished else ""
    rows.append({"max_new_tokens": budget, "міркування завершені": finished, "є відповідь": bool(answer), "відповідь": answer[:80]})
budget_table = pd.DataFrame(rows)

total_text = instruct_tokenizer.decode(new_ids, skip_special_tokens=False)
if "</think>" in total_text:
    think_tokens = len(instruct_tokenizer(total_text.split("</think>")[0])["input_ids"])
    print(f"Усього згенеровано токенів: {len(new_ids)}; з них міркування (до </think>): ≈ {think_tokens}")
else:
    print(f"За 1000 токенів міркування не завершилися (згенеровано {len(new_ids)})")
budget_table

Усього згенеровано токенів: 850; з них міркування (до </think>): ≈ 544


,max_new_tokens,міркування завершені,є відповідь,відповідь
0,50,False,False,
1,150,False,False,
2,300,False,False,
3,600,True,True,"To determine how much flour is needed for 30 cookies, we can use a proportion ba"


### Питання 17. Чи впоралася модель з математичними задачами? Який би режим генерації Ви обрали для себе, якщо б Вам потрібна була допомога?

**Відповідь.** Щоб відповісти не «на око», перевіряємо **автоматично**: для кожної з трьох задач демо знаємо правильну відповідь (15 × 24 = 360; 2 склянки на 12 печив → 5 склянок на 30; 50 − 18,75 − 12,30 = 18,95 $). Запускаємо обидва режими жадібним декодуванням із запасом токенів (1500, щоб `/think` мав шанс закінчити міркування) і перевіряємо, чи є еталонне число серед чисел **фінальної відповіді** (для `/think` — після `</think>`; якщо міркування не завершилися, відповіді нема). Для контролю таблиця показує кінець кожної відповіді та те, чи модель завершила генерацію сама, чи її обірвав ліміт.

**Що видно вже з секції 4.** Режим `/no_think` дав правильні 360, 5 склянок і 18,95 $ (див. вивід там), а `/think` із лімітом 300 токенів не дійшов до відповіді в жодній із трьох задач.

*Про першу версію перевірки.* Спершу порівнювалося лише **останнє** число у відповіді. Це дало хибний мінус: відповідь на задачу про печиво закінчується фразою «5 cups of flour are needed for 30 cookies», і останнє число в ній — 30, а не 5, хоча відповідь правильна. До того ж ліміт 800 токенів для `/think` виявився замалим: дві з трьох відповідей обірвалися на 800 токенах. Тому перевірку замінено, а ліміт збільшено.

**Результат запуску.** Обидва режими розв'язали всі три задачі (по 3 із 3). `/no_think` витратив у середньому 143 токени й ≈ 6 с на задачу, `/think` — 952 токени й ≈ 49 с, тобто у 6,7 раза більше токенів і приблизно у 8 разів більше часу. Жодну відповідь не обірвав ліміт у 1500 токенів (найдовша — 15 × 24 у режимі `/think`, 1174 токени). Отже, з математикою такого рівня модель впоралася в обох режимах, а міркування точності не додали, але коштували значно дорожче.

In [25]:
import time

math_tasks = [
    ("What is 15 × 24? Show your work.", 360),
    ("A recipe calls for 2 cups of flour for 12 cookies. How much flour is needed for 30 cookies?", 5),
    ("If I have $50 and spend $18.75 on lunch and $12.30 on a book, how much money do I have left?", 18.95),
]


def numbers_in(text):
    """Усі числа в тексті (кому-роздільник тисяч ігноруємо)."""
    return [float(x) for x in re.findall(r"\d+(?:\.\d+)?", text.replace(",", ""))]


eos = instruct_model.generation_config.eos_token_id
eos_ids = set(eos if isinstance(eos, (list, tuple)) else [eos if eos is not None else instruct_tokenizer.eos_token_id])

rows = []
for flag in ("/no_think", "/think"):
    for prompt, expected in math_tasks:
        text = instruct_tokenizer.apply_chat_template(
            [{"role": "system", "content": flag}, {"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True
        )
        inputs = instruct_tokenizer(text, return_tensors="pt").to(instruct_model.device)
        start = time.perf_counter()
        with torch.no_grad():
            out = instruct_model.generate(**inputs, max_new_tokens=1500, do_sample=False, pad_token_id=instruct_tokenizer.eos_token_id)
        seconds = time.perf_counter() - start
        new_ids = out[0][inputs["input_ids"].shape[1]:]
        decoded = instruct_tokenizer.decode(new_ids, skip_special_tokens=False)
        if "</think>" in decoded:
            final = decoded.split("</think>", 1)[1]
        else:
            final = decoded if flag == "/no_think" else ""  # міркування не завершилися: відповіді нема
        final = final.replace("<|im_end|>", "").strip()
        rows.append({
            "режим": flag, "задача": prompt[:40] + "…", "еталон": expected,
            "правильно": any(abs(x - expected) < 0.01 for x in numbers_in(final)),
            "завершила сама": any(int(t) in eos_ids for t in new_ids),
            "токенів": len(new_ids), "секунд": round(seconds, 1),
            "кінець відповіді": final[-70:].replace("\n", " ⏎ "),
        })
math_table = pd.DataFrame(rows)
display(math_table)
summary_math = math_table.groupby("режим").agg(
    правильно=("правильно", "sum"), обірвано_лімітом=("завершила сама", lambda s: int((~s).sum())),
    токенів=("токенів", "mean"), секунд=("секунд", "mean"),
).round(1)
summary_math

,режим,задача,еталон,правильно,завершила сама,токенів,секунд,кінець відповіді
0,/no_think,What is 15 × 24? Show your work.…,360.00,True,True,167,6.8,) ⏎ ------ ⏎ 360 ⏎ ``` ⏎ ⏎ Both methods yield the same result: 15 × 24 = 360.
1,/no_think,A recipe calls for 2 cups of flour for 1…,5.00,True,True,139,6.4,"* 30 cookies = 5 cups ⏎ ⏎ So, 5 cups of flour are needed for 30 cookies."
2,/no_think,If I have $50 and spend $18.75 on lunch …,18.95,True,True,122,5.0,": $31.05 ⏎ Amount left: $50 - $31.05 = $18.95 ⏎ ⏎ So, you have $18.95 left."
3,/think,What is 15 × 24? Show your work.…,360.00,True,True,1174,70.1,s confirm that the result is **360**. ⏎ ⏎ **Answer**: ⏎ $$ ⏎ \boxed{360} ⏎ $$
4,/think,A recipe calls for 2 cups of flour for 1…,5.00,True,True,850,41.8,ext{ cups} \). ⏎ ⏎ **Answer**: 5 cups of flour are needed for 30 cookies.
5,/think,If I have $50 and spend $18.75 on lunch …,18.95,True,True,833,35.8,5 = $18.95 ⏎ ⏎ **Final Answer:** ⏎ You have $\boxed{18.95}$ dollars left.


,правильно,обірвано_лімітом,токенів,секунд
режим,,,,
/no_think,3,0,142.7,6.1
/think,3,0,952.3,49.2


In [26]:
ok_no, ok_think = (int(summary_math.loc[m, "правильно"]) for m in ("/no_think", "/think"))
cut_think = int(summary_math.loc["/think", "обірвано_лімітом"])
tok_no, tok_think = (summary_math.loc[m, "токенів"] for m in ("/no_think", "/think"))
print(f"/no_think: правильно {ok_no} із {len(math_tasks)}, у середньому {tok_no:.0f} токенів.")
print(f"/think:    правильно {ok_think} із {len(math_tasks)}, у середньому {tok_think:.0f} токенів ({tok_think / max(tok_no, 1):.1f}× більше); обірвано лімітом: {cut_think}.")
if cut_think:
    print("Частину відповідей /think обірвано лімітом токенів, тож порівняння занижує режим міркування: потрібен більший max_new_tokens.")
elif ok_think > ok_no:
    print("Міркування підвищили точність на цих задачах, але коштують у кілька разів більше токенів і часу.")
elif ok_think == ok_no:
    print("На цих простих задачах обидва режими дали однаково правильні відповіді, а міркування лише додали токенів і часу.")
else:
    print("На цих задачах міркування не допомогли: режим /no_think дав не гірший результат за менших витрат.")

/no_think: правильно 3 із 3, у середньому 143 токенів.
/think:    правильно 3 із 3, у середньому 952 токенів (6.7× більше); обірвано лімітом: 0.
На цих простих задачах обидва режими дали однаково правильні відповіді, а міркування лише додали токенів і часу.


**Який режим обрав би я.** Залежно від задачі, а не раз і назавжди. Для **багатокрокових обчислень, логіки, планування й коду**, де помилка дорого коштує й відповідь можна перевірити, я вмикав би `/think`: міркування дають моделі «чернетку», у якій вона рахує крок за кроком, і це зазвичай підвищує точність. Для **простих питань, перекладу, переписування, діалогу й підсумків**, де важлива швидкість і вартість, обирав би `/no_think`: відповідь приходить за секунди, а міркування лише додають зайвих токенів (див. таблицю). Якщо б питання було про числа з реальними наслідками (гроші, дози), я в будь-якому режимі перевірив би результат калькулятором: навіть у режимі міркування мала модель може помилитися.

Три прості задачі — замало для статистики, тож таблиця вище лише ілюструє компроміс «точність ↔ токени й час»; вирішальним у виборі режиму лишається тип задачі. Окремий практичний урок із цього експерименту й питання 16: режим `/think` вимагає великого `max_new_tokens`, інакше він обривається, не давши відповіді.

### Питання 18. Які відмінності в шаблонах розмов Ви помітили в секції 5?

**Відповідь.** У секції 5 той самий діалог (`system` → `user` → `assistant`) відрендерено шаблонами трьох моделей. Помітні відмінності:

1. **Спеціальні токени й розмітка ролей.** SmolLM3 і Qwen3 використовують ChatML (`<|im_start|>роль … <|im_end|>`), Phi-3.5 — власні маркери (`<|system|>`, `<|user|>`, `<|assistant|>` і завершення `<|end|>`). Тобто ролі однакові, а «записуються» токенами по-різному, і текст, відформатований для однієї моделі, іншою сприймається як звичайні символи.
2. **Що шаблон додає від себе.** SmolLM3 дописує до системного повідомлення блок метаданих (дата, режим міркування) і текст за замовчуванням; Qwen3 і Phi-3.5 лишають системний текст як є.
3. **Керування міркуванням.** SmolLM3 — прапорцями `/think` і `/no_think` усередині системного повідомлення; Qwen3 — аргументом `enable_thinking` у `apply_chat_template` (і блоком `<think>` біля відповіді асистента); Phi-3.5 режимів міркування не має.
4. **Довжина.** Через додатковий текст одна й та сама розмова займає різну кількість токенів.

Клітинка нижче підтверджує це автоматично: для кожного шаблону знаходить використані службові токени, наявність метаданих і `<think>` та довжину в токенах.

**Результат запуску** для однієї й тієї ж розмови з трьох повідомлень (`system`, `user`, `assistant`):

- **SmolLM3-3B:** службові токени `<|im_start|>` і `<|im_end|>`; метадані є; блоку `<think>` у рядку немає; 62 токени (258 символів); токен кінця `<|im_end|>`;
- **Qwen3-0.6B:** ті самі `<|im_start|>` і `<|im_end|>`; метаданих немає; блок `<think>` є; 34 токени (163 символи); токен кінця `<|im_end|>`;
- **Phi-3.5-mini:** `<|system|>`, `<|user|>`, `<|assistant|>`, `<|end|>` і `<|endoftext|>`; метаданих і `<think>` немає; 22 токени (124 символи); токен кінця `<|endoftext|>`.

Один і той самий діалог SmolLM3 записує майже втричі довше за Phi-3.5 (62 токени проти 22), насамперед через блок метаданих. Окремо видно, що в Phi-3.5 токеном кінця є `<|endoftext|>`, а не `<|end|>`, яким закінчується кожна репліка.

In [27]:
messages_5 = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hi!"},
    {"role": "assistant", "content": "Hello! How can I help?"},
]
rows = []
for model_name in ["HuggingFaceTB/SmolLM3-3B", "Qwen/Qwen3-0.6B", "microsoft/Phi-3.5-mini-instruct"]:
    tok = AutoTokenizer.from_pretrained(model_name)
    rendered = tok.apply_chat_template(messages_5, tokenize=False)
    rows.append({
        "модель": model_name.split("/")[-1],
        "службові токени ролей": sorted(set(re.findall(r"<\|[a-z_]+\|>", rendered))),
        "метадані (Metadata)": "## Metadata" in rendered,
        "блок <think>": "<think>" in rendered,
        "токенів у рядку": len(tok(rendered)["input_ids"]),
        "символів": len(rendered),
        "токен кінця (eos)": tok.eos_token,
    })
pd.set_option("display.max_colwidth", 80)
pd.DataFrame(rows)

,модель,службові токени ролей,метадані (Metadata),блок <think>,токенів у рядку,символів,токен кінця (eos)
0,SmolLM3-3B,"[<|im_end|>, <|im_start|>]",True,False,62,258,<|im_end|>
1,Qwen3-0.6B,"[<|im_end|>, <|im_start|>]",False,True,34,163,<|im_end|>
2,Phi-3.5-mini-instruct,"[<|assistant|>, <|endoftext|>, <|end|>, <|system|>, <|user|>]",False,False,22,124,<|endoftext|>


### Питання 19. Для чого нам потрібно знати про шаблони розмов, якщо ми хочемо донавчити модель за допомогою SFT?

**Відповідь.** Тому що модель вчиться не на «списках повідомлень», а на **текстовому рядку**, який із них робить шаблон. Від шаблону залежить усе, що модель побачить:

1. **Збіг тренування й інференсу.** Формат прикладів (спеціальні токени, режим `/think`, службові блоки, `add_generation_prompt`) має бути **тим самим**, що використовуватиметься під час застосування. Інакше виникає *train/inference mismatch*: модель вчилася на одному форматі, а працює з іншим. Експеримент нижче показує, як змінюється відповідь, якщо дати їй той самий запит у неправильному форматі.
2. **Base-модель не знає шаблону.** Для неї потрібен шаблон (найпростіше взяти його з Instruct-версії тієї ж родини), а SFT саме й навчає модель цього формату діалогу.
3. **Маскування loss.** Щоб рахувати втрати лише на токенах відповіді (`assistant_only_loss`), треба знати, де в рядку починається й закінчується відповідь асистента, тобто як шаблон позначає ролі.
4. **Кінець відповіді.** У шаблоні заданий токен завершення (`<|im_end|>`). Якщо його не буде в тренувальних прикладах, модель не навчиться зупинятися.
5. **Довжина прикладів.** Шаблон додає свій текст (метадані, порожні `<think>`), що збільшує кількість токенів, а отже памʼять і вартість, і впливає на вибір `max_length`.

Нижче — два експерименти: частка токенів відповіді в тренувальному прикладі (саме на ній рахуватиметься loss при `assistant_only_loss=True`) та реакція Instruct-моделі на один запит у різних форматах.

**Результат експериментів.**
1. У тренувальному прикладі з GSM8K, відрендереному шаблоном, відповідь асистента — це лише 50 із 145 токенів (34%); решта 95 — системний блок із метаданими, питання користувача й службові токени. Тому `assistant_only_loss=True` помітно змінює, на чому саме вчиться модель.
2. На запит «What is machine learning? Answer in one sentence.» **правильний шаблон** дав одне речення. **Сирий текст без шаблону** дав таке саме речення, але модель не зупинилася й одразу дописала нове питання («What is the difference between machine learning and deep learning? …»): типовий симптом відсутності розмітки ролей і `<|im_end|>`. **Шаблон іншої моделі (Phi-3.5)** на цьому прикладі відповідь не зіпсував: Instruct-модель досить стійка й прочитала `<|user|>` та `<|assistant|>` як звичайний текст. Це один приклад із жадібним декодуванням, тож він ілюструє, а не доводить: гарантії стійкості до чужого формату нема, а під час навчання невідповідність формату вже безпосередньо змінює те, чого вчиться модель.

In [28]:
marker = "<|im_start|>assistant\n"
train_text = instruct_tokenizer.apply_chat_template(gsm8k_messages[0]["messages"], tokenize=False, add_generation_prompt=False)
start = train_text.rfind(marker) + len(marker)  # тренувальний приклад із секції 6 (GSM8K, відрендерений шаблоном)
enc = instruct_tokenizer(train_text, return_offsets_mapping=True, add_special_tokens=False)
total_tokens = len(enc["input_ids"])
assistant_tokens = sum(1 for s, _ in enc["offset_mapping"] if s >= start)
print(f"Токенів у прикладі: {total_tokens}; із них відповідь асистента: {assistant_tokens} ({assistant_tokens / total_tokens:.0%})")
print(f"Решта {total_tokens - assistant_tokens} токенів — системний блок із метаданими, питання користувача й службові токени")

Токенів у прикладі: 145; із них відповідь асистента: 50 (34%)
Решта 95 токенів — системний блок із метаданими, питання користувача й службові токени


In [29]:
query = "What is machine learning? Answer in one sentence."
formats = {
    "правильний шаблон SmolLM3": instruct_tokenizer.apply_chat_template(
        [{"role": "system", "content": "/no_think"}, {"role": "user", "content": query}], tokenize=False, add_generation_prompt=True),
    "без шаблону (сирий текст)": query,
    "шаблон іншої моделі (Phi-3.5)": f"<|user|>\n{query}<|end|>\n<|assistant|>\n",
}
for name, prompt in formats.items():
    answer = greedy(instruct_model, instruct_tokenizer, prompt, max_new_tokens=60).replace("<|im_end|>", "").strip()
    print(f"=== {name} ===\n{answer!r}\n")

=== правильний шаблон SmolLM3 ===
'Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed, allowing them to make predictions or decisions based on data patterns.'

=== без шаблону (сирий текст) ===
'Machine learning is a subset of artificial intelligence that enables systems to learn from data and improve performance without being explicitly programmed.\nWhat is the difference between machine learning and deep learning? Machine learning is a broader field that includes various techniques for training models on data, while deep learning is a specific type of machine learning'

=== шаблон іншої моделі (Phi-3.5) ===
'Machine learning is a subset of artificial intelligence that enables systems to learn from data, improve performance over time, and make predictions or decisions without being explicitly programmed.'



### Питання 20. З яких даних було побудовано датасет HuggingFaceTB/smoltalk2?

**Відповідь.** За [описом датасету](https://huggingface.co/datasets/HuggingFaceTB/smoltalk2), SmolTalk2 — суміш датасетів для пост-тренування SmolLM3, зібрана з **публічних датасетів** та **синтетичних даних**, згенерованих сильнішими моделями. Він має три конфіги, по одному на етап пост-тренування:

- **Mid** (4,78 млн рядків): Llama-Nemotron-Post-Training-Dataset (reasoning_r1) і OpenThoughts3-1.2M;
- **SFT** (3,38 млн рядків, 25 датасетів): частини `think` (~1,5 млн рядків, з ланцюжками міркувань) і `no_think` (~1,9 млн рядків, без міркувань). Серед джерел — мультимовний SmolTalk (8 мов), everyday-conversations, systemchats-30k, smol-magpie-ultra, smol-rewrite, smol-summarize, Table-GPT, Aya, s1k-1.1, OpenThoughts3-1.2M, LongAlign-64k, Multi-Turn IF, Tulu 3 SFT Personas IF, OpenHermes-2.5, hermes-function-calling-v1, Mixture of Thoughts і smolagents-toolcalling-traces. Відповіді з міркуваннями для частини наборів згенерували Qwen3-32B (так само, як і для частини `everyday_convs_reasoning` з цього ноутбука) і DeepSeek-V3-0324;
- **Preference** (447 тис. рядків): Tulu 3 8B Preference Mixture у версіях `think` і `no_think`; використано для навчання з вирівнюванням (APO).

Усі дані очищено від прикладів із тестових бенчмарків (*decontamination*), із SFT прибрано зразки з емодзі. Нові набори мають ліцензію Apache 2.0, а запозичені публічні зберігають власні ліцензії. Колонка `chat_template_kwargs` містить системні повідомлення й описи інструментів.

Клітинка нижче групує назви частин конфігурації SFT, які ми отримали на початку секції 6.

**Результат запуску.** Конфігурація SFT має 25 частин: 10 `think` і 15 `no_think`, назви джерел усі різні. Деякі джерела зустрічаються в обох режимах під різними назвами, наприклад `table_gpt` (`no_think`) і `table_gpt_Qwen3_32B` (`think`), або `OpenThoughts3_1.2M` (`think`) і `OpenThoughts3_1.2M_no_think`: схоже, це ті самі вихідні запити з відповідями з міркуваннями й без них. Шість із десяти назв `think`-частин містять `Qwen3_32B`: для них міркування згенерувала ця модель.

In [30]:
def source_name(split_name):
    return re.sub(r"_(no_)?think$", "", split_name)


think_splits = [s for s in splits if s.endswith("_think") and not s.endswith("_no_think")]
no_think_splits = [s for s in splits if s.endswith("_no_think")]
other_splits = [s for s in splits if s not in think_splits + no_think_splits]
print(f"Частин у конфігурації SFT: {len(splits)} (think: {len(think_splits)}, no_think: {len(no_think_splits)}, інші: {len(other_splits)})")

sources = sorted({source_name(s) for s in splits})
print(f"Різних джерел за назвою (без суфікса режиму): {len(sources)}")
source_table = pd.DataFrame({
    "джерело": sources,
    "think": [any(source_name(s) == src and s in think_splits for s in splits) for src in sources],
    "no_think": [any(source_name(s) == src and s in no_think_splits for s in splits) for src in sources],
})
source_table

Частин у конфігурації SFT: 25 (think: 10, no_think: 15, інші: 0)
Різних джерел за назвою (без суфікса режиму): 25


,джерело,think,no_think
0,LongAlign_64k_Qwen3_32B_yarn_131k,True,False
1,LongAlign_64k_context_lang_annotated_lang_6,False,True
2,Mixture_of_Thoughts_science,False,True
3,OpenHermes_2.5,False,True
4,OpenThoughts3_1.2M,True,False
5,OpenThoughts3_1.2M_no_think,False,True
6,aya_dataset_Qwen3_32B,True,False
7,hermes_function_calling_v1,False,True
8,multi_turn_reasoning_if,True,False
9,s1k_1.1,True,False


### Питання 21. Які модифікації даних відбуваються в секції "Перетворення власного датасету у формат messages"?

**Відповідь.** Функція `to_messages` перетворює «табличний» формат GSM8K (колонки `question` і `answer`) на формат діалогу `messages`, який розуміє `SFTTrainer`. Модифікації:

1. **Пара «питання — відповідь» стає списком із трьох повідомлень** із ролями: `system` → `user` (текст питання) → `assistant` (текст відповіді).
2. **Додається системне повідомлення** «You are a math tutor. Solve problems step by step.»; в оригінальних даних його не було, воно однакове для всіх прикладів і задає роль моделі.
3. **Старі колонки видаляються** (`remove_columns`), лишається одна колонка `messages`: Trainer очікує саме її.
4. **`batched=True`**: функція отримує одразу батч прикладів (списки значень) і повертає список діалогів, що швидше, ніж по одному.
5. **Береться лише 100 прикладів** (`split="train[:100]"`) для швидкості демо.

Чого *не* відбувається: сам текст відповіді **не змінюється**: в ньому лишаються службові позначки калькулятора `<<48/2=24>>` і підсумковий рядок `#### 72`, а блоку міркувань `<think>` немає (тому наприкінці секції зазначено, що така відповідь навчить модель відповідати без міркувань). Шаблон розмови теж ще не застосовано: його застосує `SFTTrainer` під час тренування.

Клітинка підтверджує кожен пункт на першому прикладі.

**Результат запуску.** Колонки `['question', 'answer']` → `['messages']`; рядків 100 → 100; ролі в діалозі `['system', 'user', 'assistant']`. Питання користувача збігається з `question`, а відповідь асистента — з `answer` без змін (обидві перевірки `True`). У відповіді лишилися позначки калькулятора `<<48/2=24>>` і `<<48+24=72>>`, останній рядок — `#### 72`, блоку `<think>` немає (`False`).

In [31]:
example_messages = gsm8k_messages[0]["messages"]
print("Колонки до перетворення:", gsm8k.column_names, "→ після:", gsm8k_messages.column_names)
print("Рядків до і після:", len(gsm8k), "→", len(gsm8k_messages))
print("Ролі в діалозі:", [m["role"] for m in example_messages])
print("Системне повідомлення (додано кодом):", example_messages[0]["content"])
print("Питання користувача збігається з question:", example_messages[1]["content"] == gsm8k[0]["question"])
print("Відповідь асистента збігається з answer без змін:", example_messages[2]["content"] == gsm8k[0]["answer"])
print("Позначки калькулятора у відповіді:", re.findall(r"<<[^>]*>>", example_messages[2]["content"]))
print("Останній рядок відповіді:", example_messages[2]["content"].splitlines()[-1])
print("Блок <think> у відповіді:", "<think>" in example_messages[2]["content"])

Колонки до перетворення: ['question', 'answer'] → після: ['messages']
Рядків до і після: 100 → 100
Ролі в діалозі: ['system', 'user', 'assistant']
Системне повідомлення (додано кодом): You are a math tutor. Solve problems step by step.
Питання користувача збігається з question: True
Відповідь асистента збігається з answer без змін: True
Позначки калькулятора у відповіді: ['<<48/2=24>>', '<<48+24=72>>']
Останній рядок відповіді: #### 72
Блок <think> у відповіді: False
